In [ ]:
import sys

if "google.colab" in sys.modules:
    import os

    repo = "/content/lmbike_workshop_gimbal"
    sim = "/content/lmbike_workshop_gimbal/bike_mujoco_sim"
    if not os.path.isdir(repo):
        !git clone https://github.com/innowing-robotics-interns/lmbike_workshop_gimbal.git /content/lmbike_workshop_gimbal
    %cd /content/lmbike_workshop_gimbal/bike_mujoco_sim
    if sim not in sys.path:
        sys.path.insert(0, sim)

    # Colab's Jupyter must stay the copy it shipped with. requirements.txt
    # does not list Jupyter. The constraint pins whatever Jupyter packages
    # are already installed so pip cannot replace them.
    import importlib.metadata as metadata
    protected = {
        "notebook", "jupyter", "jupyterlab", "jupyter-client", "jupyter-core",
        "jupyter-server", "ipykernel", "ipython", "nbformat", "nbclient",
        "nbconvert", "nbclassic",
    }
    pinned = []
    for dist in metadata.distributions():
        name = dist.metadata["Name"]
        if name and name.lower().replace("_", "-") in protected:
            pinned.append(f"{name}=={dist.version}")
    with open("/tmp/colab-jupyter-constraints.txt", "w", encoding="utf-8") as fh:
        fh.write("\n".join(pinned) + "\n")
    !pip install -q --upgrade-strategy only-if-needed -r requirements.txt -c /tmp/colab-jupyter-constraints.txt


# Lesson 5. Running the simulator outside the notebook

Jupyter notebooks are excellent for learning: you can run a little code, look at a graph, change a number and try again. But a finished program is not normally run from a notebook. In this last lesson you will step outside the notebook and start the same bicycle simulator as an ordinary program from a terminal, which is how you would run it for real.

Why bother? There are two main reasons.

1. **A controller has to keep running on its own.** The balancing loop has to wake up on a steady clock, 50 times a second, for as long as the bicycle is riding. This is the same 0.02-second clock that built up the I part's running total in Lesson 2. In a notebook, code only runs when you click a cell. A real controller cannot wait for clicks. It needs a program that starts, runs its clock until it is told to stop, and then ends cleanly.
2. **A notebook can quietly remember old values.** Cells can be run in any order, and more than once. So a number set in an earlier cell can still be sitting in memory and change a later result without you noticing. A program that starts fresh every time, and reads all of its numbers from a saved file, behaves the same way every time.

## Learning objectives

By the end of this lesson you will be able to:

1. Name the file that starts the simulator and the file that stores your gains.
2. Check that the gains you saved in Lesson 3 really are in `configs/tutorial.yaml`.
3. Start the simulator with `python run_sim.py` from a terminal, both with and without a window.

## Glossary

| Term | Meaning |
| --- | --- |
| Terminal | A window where you type commands instead of clicking. On Windows it is PowerShell. On macOS and Linux it is called Terminal. |
| Module | A Python file with one job, which other files can use. For example, `sim/pid_controller.py` is the module that holds the balancing loop. The `sim` folder is a collection of modules. |
| Configuration file | A text file of named settings, such as `pid_kd_roll: 11.26`. The format used here is called YAML. You can open it in any text editor. |
| Entry point | The file you start in order to run the whole program. Here it is `run_sim.py`. It loads all the other modules for you. |
| Hidden state | A value left in memory by an earlier notebook cell that still affects a later cell, even though you can no longer see where it came from. |
| Headless | Running without any window. The program does the ride, prints a short report and stops. It is useful for a quick check, and you ask for it with `--headless`. |
| Virtual environment | The private copy of Python for this project that you made in Setup 0. It is stored in the `.venv` folder. |

## Step 1. Start the lesson

As in the earlier lessons, the first code cell prepares the notebook. Run it now, and continue when you see the line that begins with `project_root:`.

In [1]:
import os
import sys

def choose_mujoco_gl(platform, display, existing):
    if existing:
        return existing
    if str(platform).startswith("linux") and not display:
        return "egl"
    return None

chosen = choose_mujoco_gl(sys.platform, os.environ.get("DISPLAY"), os.environ.get("MUJOCO_GL"))
if chosen and "MUJOCO_GL" not in os.environ:
    os.environ["MUJOCO_GL"] = chosen

def find_project_root(start=None):
    current = os.path.abspath(start or os.getcwd())
    while True:
        if os.path.isfile(os.path.join(current, "run_sim.py")) and os.path.isfile(
            os.path.join(current, "sim", "__init__.py")
        ):
            return current
        parent = os.path.dirname(current)
        if parent == current:
            break
        current = parent
    raise RuntimeError(
        "Could not find bike_mujoco_sim. Start Jupyter from the folder that "
        "contains run_sim.py, or from its tutorials folder."
    )

project_root = find_project_root()
if project_root not in sys.path:
    sys.path.insert(0, project_root)

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
from dataclasses import replace

from sim.config import load_config
from sim.loop import make_sim, run_headless, summarize
from sim.pid_controller import RollSteerPID, apply_speed_refs

np.set_printoptions(precision=4, suppress=True)
print("project_root:", project_root)
print("platform:", sys.platform, "MUJOCO_GL:", os.environ.get("MUJOCO_GL", "(unset)"))


project_root: /home/lmbike2/bike-workshop/bike_mujoco_sim
platform: linux MUJOCO_GL: egl


## Step 2. Find the project files

Up to now, the notebooks have kept the project's files out of sight. Here is a map of the important ones, all inside the `bike_mujoco_sim` folder. Notice that each file has one job. That makes it much easier to find the right place when you want to change something.

| File | What it does |
| --- | --- |
| `run_sim.py` | The entry point: the program you start from the terminal. |
| `configs/tutorial.yaml` | Your settings: the gains you tuned in Lesson 3, and the speeds. It is plain text, not a program. |
| `configs/new_bike.yaml` | A copy of the settings that is known to work. Keep this file as it is, so you always have something to compare against. Your own experiments go in `configs/tutorial.yaml`. |
| `sim/pid_controller.py` | The balancing loop. It turns the bicycle's lean into a requested steer rate, exactly as you calculated by hand in Lesson 2. |
| `sim/loop.py` | The clock. About 50 times a second it reads the bicycle's measurements, runs the balancing loop, tells the motors what to do, and moves the physics forward. |
| `sim/viewer.py` | The 3D window and the keyboard controls. |
| `sim/joystick.py` | The on-screen stick from Lesson 4, and the code that reads a gamepad if you have one. |
| `sim/models.py` | Finds the bicycle model and checks that its motors and sensors are all present. |
| `models/new_bike/new_bike.xml` | MuJoCo's description of the bicycle: the weight of each part, the wheels, and how the parts are joined. |

Run the next cell. It checks that each file in the table exists on your computer. Every line should begin with `OK`. If a line begins with `MISSING`, that file has been moved or deleted, and the commands in Step 4 will only work once it is back in place.

In [2]:
paths = [
    os.path.join(project_root, "run_sim.py"),
    os.path.join(project_root, "configs", "tutorial.yaml"),
    os.path.join(project_root, "configs", "new_bike.yaml"),
    os.path.join(project_root, "sim", "pid_controller.py"),
    os.path.join(project_root, "sim", "loop.py"),
    os.path.join(project_root, "sim", "viewer.py"),
    os.path.join(project_root, "sim", "joystick.py"),
    os.path.join(project_root, "sim", "models.py"),
    os.path.join(project_root, "models", "new_bike", "new_bike.xml"),
]
for path in paths:
    rel = os.path.relpath(path, project_root)
    print(("OK " if os.path.isfile(path) else "MISSING "), rel)


OK  run_sim.py
OK  configs/tutorial.yaml
OK  configs/new_bike.yaml
OK  sim/pid_controller.py
OK  sim/loop.py
OK  sim/viewer.py
OK  sim/joystick.py
OK  sim/models.py
OK  models/new_bike/new_bike.xml


## Step 3. Check that your gains are saved in the file

When the program runs from the terminal, it knows nothing about what happened in the notebooks. It only knows what is written in the configuration file. So before you leave the notebook, make sure the gains you chose in Lesson 3 really are saved in `configs/tutorial.yaml`.

If you completed Steps 10 and 11 of Lesson 3, they should already be there: either your own tuned gains, or the reference set. If you skipped those steps, or you are not sure, open `configs/tutorial.yaml` now, set these four lines to the reference set, and save the file:

```yaml
pid_kp_roll: 7.5356
pid_ki_roll: 1.2615
pid_kd_roll: 11.2605
pid_kp_steer: 1.1164
```

Then run the next cell. It reads the file fresh from the disk, not from the notebook's memory, and prints the four gains. They should be your own tuned gains or the reference set. If they still show 5, 0 and 6, the file has not been saved with the new values. Edit it, save it, and run the cell again.

In [3]:
tutorial_yaml = os.path.join(project_root, "configs", "tutorial.yaml")
cfg = load_config(tutorial_yaml)
print("File:", os.path.relpath(tutorial_yaml, project_root))
print(
    "pid_kp_roll={:.4f}\npid_ki_roll={:.4f}\npid_kd_roll={:.4f}\npid_kp_steer={:.4f}".format(
        cfg.pid_kp_roll, cfg.pid_ki_roll, cfg.pid_kd_roll, cfg.pid_kp_steer
    )
)


File: configs/tutorial.yaml
pid_kp_roll=5.0
pid_ki_roll=0.0
pid_kd_roll=6.0
pid_kp_steer=1.1164231977942145


## Step 4. Run the simulator from a terminal

Now start the simulator as a normal program:

1. Make sure `configs/tutorial.yaml` is saved (Step 3).
2. Open a terminal in the `bike_mujoco_sim` folder. In VS Code you can use **Terminal → New Terminal**, then use `cd` to move into the folder. You can leave Jupyter running while you do this.
3. Switch on the project's virtual environment, exactly as in Setup 0. When it is on, the start of the prompt shows `(.venv)`.
4. Type one of the `python run_sim.py …` commands below and press Enter.

Each command is the program's name followed by some **flags**. A flag is an extra word, starting with `--`, that changes how the program runs, such as which settings file to use. The three example commands do three different things:

- The first plays a ride with the speed schedule in the 3D window, as in Lessons 1 to 3.
- The second lets you drive with the stick, as in Lesson 4.
- The third runs a 6-second ride without any window and prints a short report.

Windows PowerShell:

```powershell
cd path\to\bike_mujoco_sim
.\.venv\Scripts\Activate.ps1
python run_sim.py --config configs/tutorial.yaml --model new_bike --mode speed-schedule
python run_sim.py --config configs/tutorial.yaml --model new_bike --mode remote
python run_sim.py --config configs/tutorial.yaml --model new_bike --headless --duration 6
```

macOS or Linux:

```bash
cd path/to/bike_mujoco_sim
source .venv/bin/activate
python run_sim.py --config configs/tutorial.yaml --model new_bike --mode speed-schedule
python run_sim.py --config configs/tutorial.yaml --model new_bike --mode remote
python run_sim.py --config configs/tutorial.yaml --model new_bike --headless --duration 6
```

Here is what each flag does:

| Flag | What it does |
| --- | --- |
| `--config configs/tutorial.yaml` | Chooses which settings file to load. |
| `--model new_bike` | Chooses which bicycle to load. Include it every time: without it, the program loads a different bicycle, `orange_bike`. |
| `--mode speed-schedule` | Follows the speed schedule, with no stick. This is also what happens if you leave out `--mode`. |
| `--mode remote` | Opens the 3D window together with the stick, so that you can drive. |
| `--headless` | Runs with no window, prints a short report, and stops. |
| `--duration 6` | Sets how many seconds the ride lasts. |

## Step 5. Double-check the saved gains from inside Jupyter (optional)

After you have tried the terminal commands, you can run the next cell to check, from inside the notebook, whether the gains saved in `tutorial.yaml` keep the bicycle upright for 6 seconds. It prints the same kind of report as the headless command, and finishes with the largest lean. You are looking for a largest lean under 20°.

If `tutorial.yaml` still holds weak gains (a D gain below 8), the cell says so and checks `configs/new_bike.yaml` instead, so that it still runs while you finish editing your file.

In [4]:
tutorial_yaml = os.path.join(project_root, "configs", "tutorial.yaml")
new_bike_yaml = os.path.join(project_root, "configs", "new_bike.yaml")
cfg = load_config(tutorial_yaml)
# If the student has not saved tuned gains yet, fall back so Run All still works.
if cfg.pid_kd_roll < 8.0:
    print("tutorial.yaml still weak; verifying with new_bike.yaml for this check")
    cfg = load_config(new_bike_yaml)
else:
    print("verifying tutorial.yaml")
sim = make_sim(model_spec="new_bike", mode="speed-schedule", cfg=replace(cfg, duration=6.0))
history = run_headless(sim, duration=6.0)
summarize(history, model_path=sim.model_path, mode="deployment-check")
max_roll = float(np.max(np.abs(np.degrees(history["roll"]))))
print(f"max |roll| = {max_roll:.2f} deg")
if max_roll >= 20.0:
    raise RuntimeError("saved gains did not keep the bike upright")


tutorial.yaml still weak; verifying with new_bike.yaml for this check


model:      /home/lmbike2/bike-workshop/bike_mujoco_sim/models/new_bike/new_bike.xml
mode:       deployment-check
simulated:  6.00s
max |roll|: 6.56 deg
final speed:2.183 m/s
mean |roll|: 0.64 deg
max |roll| = 6.56 deg


## Questions

These questions check that you know where your work lives once you leave the notebook. Before you run the next cell, write your answers into the two text strings at the top of it, between the quotation marks.

1. Which file do you edit to keep your gains? Write its path in `file_you_edit`.
2. Which flag runs the simulator without a window? Write it in `no_window_flag`.
3. Predict: if the program rode `configs/tutorial.yaml` exactly as it is saved right now, for 4 seconds, would the bicycle fall, or would its lean stay under 45°?

The cell tells you whether each of your two answers matches this lesson. Then it rides `configs/tutorial.yaml` exactly as it is saved on the disk, with no fallback to another file this time, and tells you whether the bicycle held or fell. If it falls, the file still has the weak starting gains. Save your tuned gains (or the reference set) and run the cell again.

In [5]:
# Edit these two strings so they match your answers, then run.
file_you_edit = ""
no_window_flag = ""

file_ok = file_you_edit.replace("\\", "/").endswith("configs/tutorial.yaml")
flag_ok = no_window_flag.strip() == "--headless"
print("file:", "that is the working copy" if file_ok else "that is not the file this lesson asks you to edit")
print("flag:", "that runs with no window" if flag_ok else "that flag is not the headless run. Read the table in Step 4.")

tutorial_yaml = os.path.join(project_root, "configs", "tutorial.yaml")
cfg = load_config(tutorial_yaml)
print(
    "saved kp={:.4f} ki={:.4f} kd={:.4f} kp_steer={:.4f}".format(
        cfg.pid_kp_roll, cfg.pid_ki_roll, cfg.pid_kd_roll, cfg.pid_kp_steer
    )
)
sim = make_sim(model_spec="new_bike", mode="speed-schedule", cfg=replace(cfg, duration=4.0))
history = run_headless(sim, duration=4.0)
roll_deg = np.degrees(np.asarray(history["roll"], dtype=float))
max_abs = float(np.max(np.abs(roll_deg)))
tip = np.where(np.abs(roll_deg) > 45.0)[0]
print(f"max |roll| = {max_abs:.1f} deg")
if len(tip) == 0:
    print("Result: held. The saved file kept roll under 45 deg.")
else:
    print("Result: fell. The saved file is still a weak set.")

file: that is the working copy
flag: that runs with no window
saved kp=5.000 ki=0.000 kd=6.000 kp_steer=1.116


max |roll| = 97.8 deg
Result: fell. The saved file is still a weak set.


## Discussion

Here are some questions students often ask after this lesson, with explanations.

### Why run the balancing loop outside the notebook at all?

Because a controller has to run by itself, on a steady clock, until it is told to stop. In a notebook, code runs only when you click a cell, cells can be run out of order, and values from earlier cells can linger as hidden state and change later results.

`run_sim.py` avoids all of that. Every time you start it, it begins fresh, reads its numbers from the configuration file, and lets `sim/loop.py` run the 50-times-a-second clock until the ride ends.

### What happens to my gains when the notebook's memory is cleared?

Anything that existed only inside the notebook is gone. That includes gains that a code cell set with `replace(...)`. The only copy that survives is the one written in a file on the disk, which is why you saved your gains in `configs/tutorial.yaml`. The terminal program reads that file, so the file is what it rides.

### What is the one command to remember?

From the `bike_mujoco_sim` folder, with `(.venv)` switched on:

```
python run_sim.py --config configs/tutorial.yaml --model new_bike
```

This starts the same simulator and the same balancing loop you have used in every lesson, with your saved gains, as a single program that you can start and stop whenever you like. Add `--headless` for a quick check without a window, or `--mode remote` to drive.